# SBI2 data staging and first-pass QC

This notebook previews the processed Altair, CellPaint-POSH, and Aconcagua files, optionally downloads them, then inspects file integrity and schemas. It does not fetch images or Aconcagua cell-level data.

Before enabling downloads, review `docs/DATASETS.md` and make sure the selected data volume has enough free space.

In [ ]:
from pathlib import Path
import os, subprocess, sys

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'scripts' / 'download_data.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Start Jupyter from this repository or open this notebook from the repository.')
DATA_DIR = Path(os.environ.get('DATA_DIR', ROOT / 'data')).expanduser().resolve()
PRESET = os.environ.get('DATA_PRESET', 'core')
RUN_DOWNLOAD = False  # Change to True only after reviewing the size preview.
print('Repository:', ROOT)
print('Data directory:', DATA_DIR)
print('Preset:', PRESET)

## 1. Preview source files and sizes

The preview checks current remote response headers and does not write dataset files.

In [ ]:
preview = subprocess.run([sys.executable, str(ROOT / 'scripts' / 'download_data.py'), '--preset', PRESET, '--data-dir', str(DATA_DIR), '--dry-run'], cwd=ROOT, text=True, capture_output=True)
print(preview.stdout)
if preview.stderr:
    print(preview.stderr)
if preview.returncode:
    raise RuntimeError(f'Dry-run failed with exit code {preview.returncode}')

## 2. Download selected datasets

Downloads are disabled by default. Set `RUN_DOWNLOAD = True` above after reviewing every listed object and its size. The script will ask for confirmation again.

In [ ]:
if RUN_DOWNLOAD:
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'download_data.py'), '--preset', PRESET, '--data-dir', str(DATA_DIR), '--yes'], cwd=ROOT, check=True)
else:
    print('Download skipped. Set RUN_DOWNLOAD = True after reviewing the preview.')

## 3. Run first-pass quality checks

Run this after downloads are complete. The report is saved under `<data-dir>/qc/quality_report.json`. It checks file hashes, H5AD/Parquet structure, metadata fields, and overlap where target labels can be identified. These checks do not prove biological validity or independently verify guide-to-gene identity.

In [ ]:
if (DATA_DIR / 'download_manifest.json').exists():
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_quality_checks.py'), '--data-dir', str(DATA_DIR), '--outdir', str(DATA_DIR / 'qc')], cwd=ROOT, check=True)
else:
    print('No download manifest found yet; run the download step first.')